In [1]:
pip install bstabdiff

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.1/590.1 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Succes

In [1]:
from bstabdiff import BlockSubunitGenerator, FeatureSpec, fit_block_subunit_generator
print("bstabdiff imported successfully")

bstabdiff imported successfully


In [2]:
import numpy as np
from bstabdiff import FeatureSpec, fit_block_subunit_generator

# Dummy HDLSS data
np.random.seed(42)
n, m = 80, 2000
X = np.random.randn(n, m).astype(np.float32)
y = np.random.randint(0, 2, size=n)
X[np.random.rand(n, m) < 0.1] = np.nan

# Feature schema
feature_specs = [FeatureSpec(name=f"f{j}", kind="continuous") for j in range(m)]

# Fit BSTabDiff
gen, train_info = fit_block_subunit_generator(
    X=X,
    feature_specs=feature_specs,
    y=y,
    M=20,
    blocks=None,
    permute_features=False,
    prior_type="diffusion",
    device="cpu",
    seed=42,
    prior_epochs=300,
    prior_batch=64,
    prior_lr=1e-3,
    verbose_every=100,
    save_dir=None,
    save_name="bstabdiff_demo",
    save_best=True,
    use_ema=True,
    ema_decay=0.999,
    return_train_info=True,
)

# Sample synthetic data
X_syn, R_syn, y_syn = gen.sample(n=50)

print("X_syn shape:", X_syn.shape)
print("R_syn shape:", R_syn.shape)
print("y_syn shape:", y_syn.shape if y_syn is not None else None)
print("Best training info:", train_info)

[prior:diffusion] epoch 1/300 | loss=1.089956
[prior:diffusion] epoch 100/300 | loss=0.555257
[prior:diffusion] epoch 200/300 | loss=0.570127
[prior:diffusion] epoch 300/300 | loss=0.566063
X_syn shape: (50, 2000)
R_syn shape: (50, 2000)
y_syn shape: (50,)
Best training info: {'best_epoch': 277, 'best_loss': 0.48506245017051697, 'best_ckpt_path': None, 'loaded_at_end': True, 'prior_type': 'diffusion', 'used_ema': True, 'ema_decay': 0.999}


In [3]:
import numpy as np
from bstabdiff import FeatureSpec, fit_block_subunit_generator

# -------------------------
# 1) Create a dummy HDLSS dataset
# -------------------------
np.random.seed(42)

n = 80    # number of samples
m = 20000   # number of features (HDLSS: n << m)

X = np.random.randn(n, m).astype(np.float32)

# binary labels
y = np.random.randint(0, 2, size=n)

# add some missingness
missing_mask = np.random.rand(n, m) < 0.1
X[missing_mask] = np.nan

# -------------------------
# 2) Define feature schema
# -------------------------
feature_specs = [
    FeatureSpec(name=f"f{j}", kind="continuous")
    for j in range(m)
]

# -------------------------
# 3) Fit BSTabDiff
# -------------------------
gen, train_info = fit_block_subunit_generator(
    X=X,
    feature_specs=feature_specs,
    y=y,
    M=20,                       # latent block dimension
    prior_type="diffusion",     # or "flow"
    device="cpu",               # change to "cuda:0" if available
    seed=42,
    prior_epochs=300,
    prior_batch=64,
    prior_lr=1e-3,
    verbose_every=100,
    save_dir=None,
    save_name="bstabdiff_demo",
    save_best=True,
    use_ema=True,
    ema_decay=0.999,
    return_train_info=True,
)

print("Training info:")
print(train_info)

# -------------------------
# 4) Sample synthetic data
# -------------------------
X_syn, R_syn, y_syn = gen.sample(n=50)

print("\nSynthetic outputs:")
print("X_syn shape:", X_syn.shape)
print("R_syn shape:", R_syn.shape)
print("y_syn shape:", None if y_syn is None else y_syn.shape)

print("\nFirst 5 synthetic labels:")
print(y_syn[:5] if y_syn is not None else None)

print("\nObserved fraction in synthetic mask:")
print(R_syn.mean())

print("\nFirst synthetic sample (first 10 features):")
print(X_syn[0, :10])

[prior:diffusion] epoch 1/300 | loss=1.087770
[prior:diffusion] epoch 100/300 | loss=0.609957
[prior:diffusion] epoch 200/300 | loss=0.565672
[prior:diffusion] epoch 300/300 | loss=0.567240
Training info:
{'best_epoch': 277, 'best_loss': 0.4772273004055023, 'best_ckpt_path': None, 'loaded_at_end': True, 'prior_type': 'diffusion', 'used_ema': True, 'ema_decay': 0.999}

Synthetic outputs:
X_syn shape: (50, 20000)
R_syn shape: (50, 20000)
y_syn shape: (50,)

First 5 synthetic labels:
[0 0 1 1 1]

Observed fraction in synthetic mask:
0.899364

First synthetic sample (first 10 features):
[-1.391971   -0.8645404   0.10322114  0.20136778  1.1143863  -1.6575179
         nan  0.95579743  0.8499241   0.7713028 ]


In [4]:
import numpy as np
from bstabdiff import (
    FeatureSpec,
    fit_block_subunit_generator,
)

# =========================================================
# 1) Create a dummy HDLSS dataset
# =========================================================
np.random.seed(42)

n = 80                    # number of samples
m = 2000                   # number of features (HDLSS: n << m)
n_classes = 2             # binary classification example
missing_rate = 0.10       # fraction of missing entries

# Real data matrix
X = np.random.randn(n, m).astype(np.float32)

# Labels
y = np.random.randint(0, n_classes, size=n)

# Add missing values
missing_mask = np.random.rand(n, m) < missing_rate
X[missing_mask] = np.nan

# =========================================================
# 2) Define feature schema
# =========================================================
# Here all features are continuous.
# For categorical features, use:
# FeatureSpec(name="f0", kind="categorical", n_categories=3)
feature_specs = [
    FeatureSpec(name=f"f{j}", kind="continuous")
    for j in range(m)
]

# =========================================================
# 3) BSTabDiff fitting configuration
# =========================================================
M = 20                         # number of latent blocks
blocks = None                  # None -> uses equal block partition automatically
permute_features = False       # whether to apply a random feature permutation
prior_type = "diffusion"       # choices: "diffusion" or "flow"
device = "cpu"                 # use "cuda:0" if GPU is available
seed = 42

# Prior training hyperparameters
prior_epochs = 300
prior_batch = 64
prior_lr = 1e-3
verbose_every = 100

# Checkpoint / best-model options
save_dir = None                # e.g. "checkpoints" to save best checkpoint
save_name = "bstabdiff_demo"
save_best = True

# EMA options (mainly useful for diffusion prior)
use_ema = True
ema_decay = 0.999

# Whether to return training diagnostics
return_train_info = True

# =========================================================
# 4) Fit BSTabDiff
# =========================================================
gen, train_info = fit_block_subunit_generator(
    X=X,
    feature_specs=feature_specs,
    y=y,
    M=M,
    blocks=blocks,
    permute_features=permute_features,
    prior_type=prior_type,
    device=device,
    seed=seed,
    prior_epochs=prior_epochs,
    prior_batch=prior_batch,
    prior_lr=prior_lr,
    verbose_every=verbose_every,
    save_dir=save_dir,
    save_name=save_name,
    save_best=save_best,
    use_ema=use_ema,
    ema_decay=ema_decay,
    return_train_info=return_train_info,
)

print("Training info:")
print(train_info)

# =========================================================
# 5) Sample synthetic data
# =========================================================
n_syn = 50

# If y is omitted, class labels are sampled automatically (when class-conditional)
X_syn, R_syn, y_syn = gen.sample(n=n_syn)

print("\nSynthetic outputs:")
print("X_syn shape:", X_syn.shape)   # synthetic data
print("R_syn shape:", R_syn.shape)   # observed/missing mask (1=observed, 0=missing)
print("y_syn shape:", None if y_syn is None else y_syn.shape)

print("\nFirst 5 synthetic labels:")
print(y_syn[:5] if y_syn is not None else None)

print("\nObserved fraction in synthetic mask:")
print(R_syn.mean())

print("\nFirst synthetic sample (first 10 features):")
print(X_syn[0, :10])

# =========================================================
# 6) Optional: sample from a fixed class
# =========================================================
X_syn_c1, R_syn_c1, y_syn_c1 = gen.sample(n=10, y=1)

print("\nFixed-class synthetic labels:")
print(y_syn_c1)

# =========================================================
# 7) Optional: inspect learned model structure
# =========================================================
print("\nModel summary:")
print("Number of features (m):", gen.m)
print("Number of blocks (M):", gen.M)
print("Prior type:", gen.prior_type)
print("Has permutation:", gen.perm is not None)

[prior:diffusion] epoch 1/300 | loss=1.089956
[prior:diffusion] epoch 100/300 | loss=0.555257
[prior:diffusion] epoch 200/300 | loss=0.570127
[prior:diffusion] epoch 300/300 | loss=0.566063
Training info:
{'best_epoch': 277, 'best_loss': 0.48506245017051697, 'best_ckpt_path': None, 'loaded_at_end': True, 'prior_type': 'diffusion', 'used_ema': True, 'ema_decay': 0.999}

Synthetic outputs:
X_syn shape: (50, 2000)
R_syn shape: (50, 2000)
y_syn shape: (50,)

First 5 synthetic labels:
[0 0 1 1 1]

Observed fraction in synthetic mask:
0.89968

First synthetic sample (first 10 features):
[-0.02387629 -0.32644042 -0.8905753  -1.3320501  -1.8623976   0.3787238
         nan -1.1465378  -1.6162736  -0.35329098]

Fixed-class synthetic labels:
[1 1 1 1 1 1 1 1 1 1]

Model summary:
Number of features (m): 2000
Number of blocks (M): 20
Prior type: diffusion
Has permutation: False


In [5]:
pip show bstabdiff

Name: bstabdiff
Version: 0.1.0
Summary: BSTabDiff: Block-Subunit Diffusion Priors for High-Dimensional Tabular Data Generation
Home-page: https://github.com/zadid6pretam/BSTabDiff
Author: Al Zadid Sultan Bin Habib, Md Younus Ahamed, Prashnna Kumar Gyawali, Gianfranco Doretto, Donald A. Adjeroh
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: catboost, numpy, pandas, scikit-learn, tabpfn, torch
Required-by: 


In [6]:

from importlib.metadata import version

print("BSTabDiff version:", version("bstabdiff"))

BSTabDiff version: 0.1.0
